# 2025 NYC Yellow Taxi — Data Cleaning

SI 671 Final Project | Shared Data Preparation

## Objectives
1. Validate trip timestamps and taxi zone IDs.
2. Identify missing, invalid, and anomalous records.
3. Apply reproducible cleaning rules.
4. Preserve valid trips for network mining and forecasting.
5. Generate cleaning reports and local Parquet outputs.

Raw and processed datasets are stored locally.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find project root
cwd = Path.cwd().resolve()

if (cwd / "data").is_dir():
    ROOT = cwd
elif (cwd.parent / "data").is_dir():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Cannot locate project root.")

RAW_DIR = ROOT / "data" / "raw"
EXTERNAL_DIR = ROOT / "data" / "external"
PROCESSED_DIR = ROOT / "data" / "processed"

print("Project root:", ROOT)

Project root: /Users/wushixundelaopo/Desktop/Urban_Mobility_Mining


In [3]:
raw_path = RAW_DIR / "yellow_tripdata_2025-01.parquet"
zone_path = EXTERNAL_DIR / "taxi_zone_lookup.csv"

assert raw_path.exists(), f"File not found: {raw_path}"
assert zone_path.exists(), f"File not found: {zone_path}"

df = pd.read_parquet(raw_path)
zones = pd.read_csv(zone_path)

print("Raw trips:", f"{len(df):,}")
print("Shape:", df.shape)

display(df.head())
display(df.dtypes)

Raw trips: 3,475,226
Shape: (3475226, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [4]:
pickup_col = "tpep_pickup_datetime"
dropoff_col = "tpep_dropoff_datetime"

required_cols = [
    pickup_col,
    dropoff_col,
    "PULocationID",
    "DOLocationID",
    "trip_distance",
]

missing_cols = set(required_cols) - set(df.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"

pickup = pd.to_datetime(df[pickup_col], errors="coerce")
dropoff = pd.to_datetime(df[dropoff_col], errors="coerce")

duration_min = (
    dropoff - pickup
).dt.total_seconds() / 60

valid_zone_ids = set(zones["LocationID"].dropna())

checks = {
    "Missing pickup time": pickup.isna(),
    "Missing dropoff time": dropoff.isna(),
    "Pickup outside 2025": (
        pickup.notna()
        & (
            (pickup < pd.Timestamp("2025-01-01"))
            | (pickup >= pd.Timestamp("2026-01-01"))
        )
    ),
    "Non-positive duration": duration_min.le(0),
    "Invalid pickup zone": ~df["PULocationID"].isin(valid_zone_ids),
    "Invalid dropoff zone": ~df["DOLocationID"].isin(valid_zone_ids),
    "Negative distance": df["trip_distance"].lt(0),
    "Zero distance": df["trip_distance"].eq(0),
    "Same pickup/dropoff zone": (
        df["PULocationID"] == df["DOLocationID"]
    ),
}

quality_report = pd.DataFrame({
    "Issue": list(checks.keys()),
    "Count": [mask.sum() for mask in checks.values()],
})

quality_report["Percentage"] = (
    quality_report["Count"] / len(df) * 100
).round(3)

display(quality_report)

,Issue,Count,Percentage
0,Missing pickup time,0,0.000
1,Missing dropoff time,0,0.000
2,Pickup outside 2025,21,0.001
3,Non-positive duration,2051,0.059
4,Invalid pickup zone,0,0.000
5,Invalid dropoff zone,0,0.000
6,Negative distance,0,0.000
7,Zero distance,90893,2.615
8,Same pickup/dropoff zone,180506,5.194


In [5]:
diagnostics = pd.DataFrame({
    "trip_distance": df["trip_distance"],
    "duration_min": duration_min
})

display(
    diagnostics.describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99, 0.999]
    )
)

,trip_distance,duration_min
count,3.475226e+06,3.475226e+06
mean,5.855126e+00,1.501812e+01
std,5.646016e+02,3.871358e+01
min,0.000000e+00,-5.147232e+04
1%,0.000000e+00,6.666667e-01
5%,3.900000e-01,3.366667e+00
50%,1.670000e+00,1.170000e+01
95%,1.183000e+01,3.620000e+01
99%,1.950000e+01,5.891667e+01
99.9%,2.885775e+01,9.826292e+01


## Extreme-value diagnostics

Investigate unusually large trip distances and durations
before defining the final cleaning thresholds.

In [6]:
# Diagnostic thresholds only — not final cleaning rules

thresholds = {
    "distance_gt_50": df["trip_distance"] > 50,
    "distance_gt_100": df["trip_distance"] > 100,
    "duration_gt_180": duration_min > 180,
    "duration_gt_360": duration_min > 360,
}

extreme_report = pd.DataFrame({
    "Issue": list(thresholds.keys()),
    "Count": [mask.sum() for mask in thresholds.values()],
})

extreme_report["Percentage"] = (
    extreme_report["Count"] / len(df) * 100
).round(4)

display(extreme_report)

,Issue,Count,Percentage
0,distance_gt_50,533,0.0153
1,distance_gt_100,162,0.0047
2,duration_gt_180,1377,0.0396
3,duration_gt_360,1203,0.0346


In [7]:
# Diagnostic thresholds only — not final cleaning rules

thresholds = {
    "distance_gt_50": df["trip_distance"] > 50,
    "distance_gt_100": df["trip_distance"] > 100,
    "duration_gt_180": duration_min > 180,
    "duration_gt_360": duration_min > 360,
}

extreme_report = pd.DataFrame({
    "Issue": list(thresholds.keys()),
    "Count": [mask.sum() for mask in thresholds.values()],
})

extreme_report["Percentage"] = (
    extreme_report["Count"] / len(df) * 100
).round(4)

display(extreme_report)

,Issue,Count,Percentage
0,distance_gt_50,533,0.0153
1,distance_gt_100,162,0.0047
2,duration_gt_180,1377,0.0396
3,duration_gt_360,1203,0.0346


## Apply preliminary cleaning rules

Apply the shared data cleaning criteria and evaluate
the number of retained records before exporting data.

In [10]:
# Preliminary cleaning rules for January 2025

# January 2025 pickup window
month_start = pd.Timestamp("2025-01-01")
month_end = pd.Timestamp("2025-02-01")

valid_pickup = (
    pickup.notna()
    & (pickup >= month_start)
    & (pickup < month_end)
)

valid_dropoff = dropoff.notna()

valid_duration = (
    duration_min.notna()
    & (duration_min > 0)
    & (duration_min <= 360)
)

valid_locations = (
    df["PULocationID"].isin(valid_zone_ids)
    & df["DOLocationID"].isin(valid_zone_ids)
)

valid_distance = (
    df["trip_distance"].notna()
    & (df["trip_distance"] >= 0)
    & (df["trip_distance"] <= 100)
)

keep_mask = (
    valid_pickup
    & valid_dropoff
    & valid_duration
    & valid_locations
    & valid_distance
)

df_clean = df.loc[keep_mask].copy()

print(f"Original trips: {len(df):,}")
print(f"Retained trips: {len(df_clean):,}")
print(f"Removed trips: {(~keep_mask).sum():,}")
print(f"Retention rate: {keep_mask.mean():.4%}")

Original trips: 3,475,226
Retained trips: 3,471,790
Removed trips: 3,436
Retention rate: 99.9011%


In [12]:
# Check pickup timestamps outside the target month

outside_month = (
    pickup.notna()
    & (
        (pickup < month_start)
        | (pickup >= month_end)
    )
)

print(
    "Pickup records outside target month:",
    outside_month.sum()
)

display(
    df.loc[
        outside_month,
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID",
            "DOLocationID",
        ],
    ].head(10)
)

Pickup records outside target month: 22


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID
605,2024-12-31 23:30:03,2024-12-31 23:43:02,246,13
687,2024-12-31 23:31:38,2024-12-31 23:41:48,43,140
688,2024-12-31 23:46:38,2025-01-01 00:03:03,229,24
861,2024-12-31 23:56:19,2025-01-01 00:11:19,68,107
1108,2024-12-31 23:55:37,2025-01-01 00:01:26,56,56
1312,2024-12-31 23:52:40,2025-01-01 00:23:03,114,151
2276,2024-12-31 23:49:24,2024-12-31 23:57:30,142,151
3941,2024-12-31 23:27:13,2024-12-31 23:35:48,170,141
3942,2024-12-31 23:37:42,2024-12-31 23:43:10,229,141
4324,2024-12-31 23:51:20,2025-01-01 00:00:00,138,40


## Post-Cleaning Validation

Verify that the cleaned dataset satisfies the defined
data quality requirements before exporting Parquet files.

In [13]:
# Validate January cleaned dataset

clean_pickup = pd.to_datetime(
    df_clean["tpep_pickup_datetime"],
    errors="coerce"
)

clean_dropoff = pd.to_datetime(
    df_clean["tpep_dropoff_datetime"],
    errors="coerce"
)

clean_duration = (
    clean_dropoff - clean_pickup
).dt.total_seconds() / 60

validation_checks = {
    "Missing pickup time": clean_pickup.isna().sum() == 0,
    "Missing dropoff time": clean_dropoff.isna().sum() == 0,
    "Pickup within target month": (
        (clean_pickup >= month_start)
        & (clean_pickup < month_end)
    ).all(),
    "Duration within (0, 360] minutes": (
        (clean_duration > 0)
        & (clean_duration <= 360)
    ).all(),
    "Valid pickup zones": (
        df_clean["PULocationID"].isin(valid_zone_ids)
    ).all(),
    "Valid dropoff zones": (
        df_clean["DOLocationID"].isin(valid_zone_ids)
    ).all(),
    "Distance within [0, 100] miles": (
        df_clean["trip_distance"].notna()
        & (df_clean["trip_distance"] >= 0)
        & (df_clean["trip_distance"] <= 100)
    ).all(),
}

validation_report = pd.DataFrame({
    "Check": list(validation_checks.keys()),
    "Passed": list(validation_checks.values()),
})

display(validation_report)

assert validation_report["Passed"].all(), (
    "Some cleaning validation checks failed!"
)

print("All cleaning validation checks passed.")
print(f"Final January records: {len(df_clean):,}")

,Check,Passed
0,Missing pickup time,True
1,Missing dropoff time,True
2,Pickup within target month,True
3,"Duration within (0, 360] minutes",True
4,Valid pickup zones,True
5,Valid dropoff zones,True
6,"Distance within [0, 100] miles",True


All cleaning validation checks passed.
Final January records: 3,471,790


In [14]:
# Cleaning summary

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original records",
        "Retained records",
        "Removed records",
        "Removal rate (%)",
    ],
    "Value": [
        len(df),
        len(df_clean),
        len(df) - len(df_clean),
        round((1 - len(df_clean) / len(df)) * 100, 4),
    ],
})

display(cleaning_summary)

,Metric,Value
0,Original records,3.475226e+06
1,Retained records,3.471790e+06
2,Removed records,3.436000e+03
3,Removal rate (%),9.890000e-02


## Full-Year Data Cleaning

Apply a consistent cleaning pipeline to all 12 monthly
Parquet files and generate reproducible quality reports.

In [23]:


def clean_month(year, month, save=True):
    """
    Clean monthly NYC Yellow Taxi data.

    Retain records with valid pickup timestamps and pickup zones.
    Flag other data-quality issues without automatically dropping trips.
    """

    month_label = f"{year}-{month:02d}"
    month_start = pd.Timestamp(year=year, month=month, day=1)
    month_end = month_start + pd.DateOffset(months=1)

    input_path = RAW_DIR / f"yellow_tripdata_{month_label}.parquet"
    output_dir = PROCESSED_DIR / "cleaned_trips"
    output_path = output_dir / f"cleaned_{month_label}.parquet"

    if not input_path.exists():
        raise FileNotFoundError(input_path)

    columns = [
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
    ]

    trips = pd.read_parquet(input_path, columns=columns)
    original_count = len(trips)

    # 1. Parse timestamps
    pu = pd.to_datetime(
        trips["tpep_pickup_datetime"], errors="coerce"
    )
    do = pd.to_datetime(
        trips["tpep_dropoff_datetime"], errors="coerce"
    )

    duration = (do - pu).dt.total_seconds() / 60

    # 2. Minimum validity requirements
    valid_pickup_time = (
        pu.notna()
        & pu.ge(month_start)
        & pu.lt(month_end)
    )

    valid_pickup_zone = (
        trips["PULocationID"].isin(valid_zone_ids)
    )

    keep = valid_pickup_time & valid_pickup_zone

    # 3. Preserve valid pickup records
    cleaned = trips.loc[keep].copy()

    cleaned["trip_duration_min"] = duration.loc[keep]

    # 4. Add data-quality flags
    cleaned["valid_dropoff_zone"] = (
        cleaned["DOLocationID"].isin(valid_zone_ids)
    )

    cleaned["valid_duration"] = (
        cleaned["trip_duration_min"].notna()
        & cleaned["trip_duration_min"].gt(0)
        & cleaned["trip_duration_min"].le(360)
    )

    cleaned["valid_distance"] = (
        cleaned["trip_distance"].notna()
        & cleaned["trip_distance"].ge(0)
        & cleaned["trip_distance"].le(100)
    )

    cleaned["zero_duration"] = (
        cleaned["trip_duration_min"].eq(0)
    )

    cleaned["negative_duration"] = (
        cleaned["trip_duration_min"].lt(0)
    )

    cleaned["valid_od"] = (
        cleaned["valid_dropoff_zone"]
        & cleaned["valid_duration"]
        & cleaned["valid_distance"]
    )

    # 5. Post-cleaning validation
    assert len(cleaned) == int(keep.sum())
    assert cleaned["tpep_pickup_datetime"].notna().all()

    assert (
        cleaned["tpep_pickup_datetime"].ge(month_start)
    ).all()

    assert (
        cleaned["tpep_pickup_datetime"].lt(month_end)
    ).all()

    assert (
        cleaned["PULocationID"].isin(valid_zone_ids)
    ).all()

    assert not cleaned[
        [
            "valid_dropoff_zone",
            "valid_duration",
            "valid_distance",
            "zero_duration",
            "negative_duration",
            "valid_od",
        ]
    ].isna().any().any()

    # 6. Generate cleaning report
    report = {
        "Month": month_label,
        "Original": original_count,
        "Retained": len(cleaned),
        "Removed": int((~keep).sum()),
        "Removal_pct": round((~keep).mean() * 100, 4),

        "Outside_month": int(
            (
                pu.notna()
                & (
                    pu.lt(month_start)
                    | pu.ge(month_end)
                )
            ).sum()
        ),

        "Invalid_pickup_zone": int(
            (~valid_pickup_zone).sum()
        ),

        "Zero_duration_retained": int(
            cleaned["zero_duration"].sum()
        ),

        "Negative_duration_retained": int(
            cleaned["negative_duration"].sum()
        ),

        "Invalid_duration_retained": int(
            (~cleaned["valid_duration"]).sum()
        ),

        "Invalid_distance_retained": int(
            (~cleaned["valid_distance"]).sum()
        ),

        "Invalid_dropoff_zone_retained": int(
            (~cleaned["valid_dropoff_zone"]).sum()
        ),

        "OD_valid_records": int(
            cleaned["valid_od"].sum()
        ),
    }

    # 7. Save locally
    if save:
        output_dir.mkdir(parents=True, exist_ok=True)
        cleaned.to_parquet(output_path, index=False)

    return report


In [25]:
# Test updated cleaning policy on January 2025

jan_report = clean_month(2025, 1, save=False)

display(pd.DataFrame([jan_report]).T)

assert jan_report["Original"] == 3_475_226
assert jan_report["Retained"] == 3_475_204
assert jan_report["Removed"] == 22

assert (
    jan_report["Original"]
    == jan_report["Retained"] + jan_report["Removed"]
)

print("January cleaning validation passed!")

,0
Month,2025-01
Original,3475226
Retained,3475204
Removed,22
Removal_pct,0.0006
Outside_month,22
Invalid_pickup_zone,0
Zero_duration_retained,1927
Negative_duration_retained,124
Invalid_duration_retained,3254


January cleaning validation passed!


In [26]:
monthly_reports = []

for month in range(1, 13):
    report = clean_month(2025, month, save=True)
    monthly_reports.append(report)

    print(
        f"{report['Month']}: "
        f"{report['Retained']:,} retained, "
        f"{report['Removal_pct']:.4f}% removed"
    )

full_report = pd.DataFrame(monthly_reports)

display(full_report)

print("\nFULL-YEAR SUMMARY")
print("Original:", f"{full_report['Original'].sum():,}")
print("Retained:", f"{full_report['Retained'].sum():,}")
print("Removed:", f"{full_report['Removed'].sum():,}")
print("OD-valid:", f"{full_report['OD_valid_records'].sum():,}")

2025-01: 3,475,204 retained, 0.0006% removed
2025-02: 3,577,512 retained, 0.0009% removed
2025-03: 4,145,224 retained, 0.0008% removed
2025-04: 3,970,546 retained, 0.0002% removed
2025-05: 4,591,821 retained, 0.0005% removed
2025-06: 4,322,940 retained, 0.0005% removed
2025-07: 3,898,956 retained, 0.0002% removed
2025-08: 3,574,074 retained, 0.0005% removed
2025-09: 4,251,008 retained, 0.0002% removed
2025-10: 4,428,686 retained, 0.0003% removed
2025-11: 4,181,420 retained, 0.0006% removed
2025-12: 4,304,997 retained, 0.0002% removed


,Month,Original,Retained,Removed,Removal_pct,Outside_month,Invalid_pickup_zone,Zero_duration_retained,Negative_duration_retained,Invalid_duration_retained,Invalid_distance_retained,Invalid_dropoff_zone_retained,OD_valid_records
0,2025-01,3475226,3475204,22,0.0006,22,0,1927,124,3254,162,0,3471790
1,2025-02,3577543,3577512,31,0.0009,31,0,5071,93,6297,172,0,3571047
2,2025-03,4145257,4145224,33,0.0008,33,0,22199,81,23653,214,0,4121366
3,2025-04,3970553,3970546,7,0.0002,7,0,34606,163,36153,306,0,3934096
4,2025-05,4591845,4591821,24,0.0005,24,0,64171,101,65585,370,0,4525880
5,2025-06,4322960,4322940,20,0.0005,20,0,68352,231,69758,264,0,4252949
6,2025-07,3898963,3898956,7,0.0002,7,0,56063,1,56998,274,0,3841720
7,2025-08,3574091,3574074,17,0.0005,17,0,47831,2,48861,245,0,3525004
8,2025-09,4251015,4251008,7,0.0002,7,0,57179,0,58388,214,0,4192431
9,2025-10,4428699,4428686,13,0.0003,13,0,67966,2,69221,229,0,4359267



FULL-YEAR SUMMARY
Original: 48,722,602
Retained: 48,722,388
Removed: 214
OD-valid: 48,158,529


## Full-Year Cleaning Diagnostics

Investigate the distribution of cleaning violations across months
before finalizing the shared cleaning policy.

In [18]:

# Compare cleaning issues across all 12 months

diagnostic_cols = [
    "Month",
    "Original",
    "Removed",
    "Removal_pct",
    "Outside_month",
    "Non_positive_duration",
    "Duration_gt_360",
    "Distance_gt_100",
]

diagnostic_report = full_report[diagnostic_cols].copy()

for col in diagnostic_cols[4:]:
    diagnostic_report[col + "_pct"] = (
        diagnostic_report[col] /
        diagnostic_report["Original"] * 100
    ).round(4)

display(diagnostic_report.T)


,0,1,2,3,4,5,6,7,8,9,10,11
Month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
Original,3475226,3577543,4145257,3970553,4591845,4322960,3898963,3574091,4251015,4428699,4181444,4305006
Removed,3436,6496,23891,36457,65965,70011,57243,49087,58584,69432,63705,59766
Removal_pct,0.0989,0.1816,0.5763,0.9182,1.4366,1.6195,1.4682,1.3734,1.3781,1.5678,1.5235,1.3883
Outside_month,22,31,33,7,24,20,7,17,7,13,24,9
Non_positive_duration,2051,5164,22280,34769,64272,68583,56064,47833,57179,67968,62120,58021
Duration_gt_360,1203,1133,1373,1384,1313,1175,934,1028,1209,1254,1363,1547
Distance_gt_100,162,172,214,306,370,264,274,245,214,229,223,197
Outside_month_pct,0.0006,0.0009,0.0008,0.0002,0.0005,0.0005,0.0002,0.0005,0.0002,0.0003,0.0006,0.0002
Non_positive_duration_pct,0.059,0.1443,0.5375,0.8757,1.3997,1.5865,1.4379,1.3383,1.3451,1.5347,1.4856,1.3478


## Investigate Non-positive Trip Durations

Determine whether the high removal rates are primarily caused
by zero-duration trips or negative-duration records.

In [19]:

# Investigate non-positive durations across all months

duration_reports = []

for month in range(1, 13):
    month_label = f"2025-{month:02d}"
    path = RAW_DIR / f"yellow_tripdata_{month_label}.parquet"

    temp = pd.read_parquet(
        path,
        columns=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "PULocationID",
            "DOLocationID",
        ],
    )

    pu = pd.to_datetime(
        temp["tpep_pickup_datetime"], errors="coerce"
    )
    do = pd.to_datetime(
        temp["tpep_dropoff_datetime"], errors="coerce"
    )

    duration = (do - pu).dt.total_seconds() / 60

    zero = duration == 0
    negative = duration < 0
    non_positive = zero | negative

    duration_reports.append({
        "Month": month_label,
        "Zero_duration": int(zero.sum()),
        "Negative_duration": int(negative.sum()),
        "Non_positive_total": int(non_positive.sum()),
        "Zero_with_positive_distance": int(
            (zero & (temp["trip_distance"] > 0)).sum()
        ),
        "Negative_with_positive_distance": int(
            (negative & (temp["trip_distance"] > 0)).sum()
        ),
        "Non_positive_pct": round(
            non_positive.mean() * 100, 4
        ),
    })

    del temp

duration_report = pd.DataFrame(duration_reports)

display(duration_report)

print("\nFULL-YEAR TOTALS")
print(
    duration_report.drop(columns=["Month"]).sum(
        numeric_only=True
    )
)


,Month,Zero_duration,Negative_duration,Non_positive_total,Zero_with_positive_distance,Negative_with_positive_distance,Non_positive_pct
0,2025-01,1927,124,2051,1235,124,0.0590
1,2025-02,5071,93,5164,4405,93,0.1443
2,2025-03,22199,81,22280,21437,81,0.5375
3,2025-04,34606,163,34769,33770,162,0.8757
4,2025-05,64171,101,64272,62649,100,1.3997
5,2025-06,68352,231,68583,66338,231,1.5865
6,2025-07,56063,1,56064,54554,1,1.4379
7,2025-08,47831,2,47833,46166,2,1.3383
8,2025-09,57179,0,57179,55976,0,1.3451
9,2025-10,67966,2,67968,66633,0,1.5347



FULL-YEAR TOTALS
Zero_duration                      544069.0000
Negative_duration                    2235.0000
Non_positive_total                 546304.0000
Zero_with_positive_distance        528530.0000
Negative_with_positive_distance      2223.0000
Non_positive_pct                       13.0921
dtype: float64


In [20]:

# Inspect zero-duration records in January and June

for month in [1, 6]:
    path = RAW_DIR / f"yellow_tripdata_2025-{month:02d}.parquet"
    sample_df = pd.read_parquet(path)

    pu = pd.to_datetime(
        sample_df["tpep_pickup_datetime"], errors="coerce"
    )
    do = pd.to_datetime(
        sample_df["tpep_dropoff_datetime"], errors="coerce"
    )

    zero_mask = (do == pu) & pu.notna()

    print(f"\n2025-{month:02d}")
    print("Zero-duration records:", zero_mask.sum())

    columns = [
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "total_amount",
    ]

    available = [
        col for col in columns if col in sample_df.columns
    ]

    display(
        sample_df.loc[zero_mask, available].head(15)
    )

    if "VendorID" in sample_df.columns:
        vendor_report = pd.crosstab(
            sample_df["VendorID"],
            zero_mask,
            normalize="index"
        ) * 100

        display(vendor_report.round(3))

    del sample_df



2025-01
Zero-duration records: 1927


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount
92,1,2025-01-01 00:49:48,2025-01-01 00:49:48,87,264,0.00,20.06,20.06
11604,1,2025-01-01 01:42:36,2025-01-01 01:42:36,158,264,0.00,3.00,8.00
13820,1,2025-01-01 02:13:25,2025-01-01 02:13:25,233,264,0.00,114.00,114.00
14605,1,2025-01-01 02:09:52,2025-01-01 02:09:52,237,264,0.00,3.00,8.00
14607,1,2025-01-01 02:49:40,2025-01-01 02:49:40,162,264,0.00,3.00,8.00
16093,1,2025-01-01 02:21:20,2025-01-01 02:21:20,164,264,0.00,16.30,21.30
19599,1,2025-01-01 03:46:16,2025-01-01 03:46:16,36,264,0.00,31.91,31.91
31891,7,2025-01-01 12:44:07,2025-01-01 12:44:07,143,230,1.56,17.00,31.00
34038,1,2025-01-01 12:21:49,2025-01-01 12:21:49,148,264,0.00,14.20,18.20
34951,1,2025-01-01 12:53:45,2025-01-01 12:53:45,164,264,0.00,21.55,21.55


col_0,False,True
VendorID,,
1,99.915,0.085
2,99.997,0.003
6,98.978,1.022
7,0.000,100.000



2025-06
Zero-duration records: 68352


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount
4,7,2025-06-01 00:16:32,2025-06-01 00:16:32,48,234,2.22,20.5,31.50
25,7,2025-06-01 00:07:41,2025-06-01 00:07:41,186,79,2.21,17.0,24.25
26,7,2025-06-01 00:16:41,2025-06-01 00:16:41,79,114,0.56,5.1,13.02
27,7,2025-06-01 00:32:06,2025-06-01 00:32:06,114,186,2.01,14.2,19.95
48,7,2025-06-01 00:00:09,2025-06-01 00:00:09,158,162,2.84,21.2,32.34
68,7,2025-06-01 00:16:17,2025-06-01 00:16:17,164,158,1.58,11.4,17.15
69,7,2025-06-01 00:41:18,2025-06-01 00:41:18,158,238,4.99,26.8,39.06
123,7,2025-06-01 00:54:09,2025-06-01 00:54:09,48,246,1.35,9.3,17.05
163,7,2025-06-01 00:03:00,2025-06-01 00:03:00,162,230,1.26,10.7,19.74
164,7,2025-06-01 00:38:06,2025-06-01 00:38:06,48,255,6.62,33.1,46.62


col_0,False,True
VendorID,,
1,99.914,0.086
2,99.998,0.002
6,99.604,0.396
7,0.000,100.000


In [21]:
for month in [1, 6]:
    path = RAW_DIR / f"yellow_tripdata_2025-{month:02d}.parquet"

    sample = pd.read_parquet(
        path,
        columns=[
            "VendorID",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID",
            "DOLocationID",
            "trip_distance",
        ],
    )

    helix = sample[sample["VendorID"] == 7].copy()

    print(f"\n2025-{month:02d}")
    print("Helix records:", len(helix))

    if len(helix):
        print(
            "Zero-duration percentage:",
            (
                helix["tpep_pickup_datetime"]
                == helix["tpep_dropoff_datetime"]
            ).mean() * 100
        )

        duplicate_cols = [
            "tpep_pickup_datetime",
            "PULocationID",
            "DOLocationID",
            "trip_distance",
        ]

        print(
            "Exact match on selected fields (%):",
            helix.duplicated(
                subset=duplicate_cols,
                keep=False
            ).mean() * 100
        )

        print(
            "Unique pickup timestamps:",
            helix["tpep_pickup_datetime"].nunique()
        )


2025-01
Helix records: 1206
Zero-duration percentage: 100.0
Exact match on selected fields (%): 0.0
Unique pickup timestamps: 1206

2025-06
Helix records: 67573
Zero-duration percentage: 100.0
Exact match on selected fields (%): 0.0
Unique pickup timestamps: 66383


## Final Validation of Cleaned Data

Verify that all 12 cleaned Parquet files exist, contain the
expected columns, and match the monthly cleaning reports.

In [27]:

import pyarrow.parquet as pq

# Validate saved monthly Parquet files
validation_results = []

required_columns = {
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "trip_duration_min",
    "valid_dropoff_zone",
    "valid_duration",
    "valid_distance",
    "zero_duration",
    "negative_duration",
    "valid_od",
}

for month in range(1, 13):
    label = f"2025-{month:02d}"
    path = PROCESSED_DIR / "cleaned_trips" / f"cleaned_{label}.parquet"

    assert path.exists(), f"Missing file: {path}"

    parquet = pq.ParquetFile(path)
    columns = set(parquet.schema_arrow.names)
    saved_rows = parquet.metadata.num_rows

    expected_rows = int(
        full_report.loc[
            full_report["Month"] == label, "Retained"
        ].iloc[0]
    )

    validation_results.append({
        "Month": label,
        "File_exists": True,
        "Schema_valid": required_columns.issubset(columns),
        "Rows_match": saved_rows == expected_rows,
        "Saved_rows": saved_rows,
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

assert validation_df["Schema_valid"].all()
assert validation_df["Rows_match"].all()

print("All 12 files passed metadata validation.")
print(f"Total saved records: {validation_df['Saved_rows'].sum():,}")


,Month,File_exists,Schema_valid,Rows_match,Saved_rows
0,2025-01,True,True,True,3475204
1,2025-02,True,True,True,3577512
2,2025-03,True,True,True,4145224
3,2025-04,True,True,True,3970546
4,2025-05,True,True,True,4591821
5,2025-06,True,True,True,4322940
6,2025-07,True,True,True,3898956
7,2025-08,True,True,True,3574074
8,2025-09,True,True,True,4251008
9,2025-10,True,True,True,4428686


All 12 files passed metadata validation.
Total saved records: 48,722,388


In [28]:
report_dir = PROCESSED_DIR / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / "cleaning_report_2025.csv"

full_report.to_csv(report_path, index=False)

print("Cleaning report saved:", report_path)

Cleaning report saved: /Users/wushixundelaopo/Desktop/Urban_Mobility_Mining/data/processed/reports/cleaning_report_2025.csv
